# 01 — Exploratory Data Analysis
**Loan Default Prediction Project**

Goal: understand the dataset structure, distributions, missing values, and relationships between features and the target variable.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pointbiserialr
import warnings, sys, os

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

RANDOM_STATE = 42
PLOTS_DIR = '../plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 1. Load & Inspect

In [ ]:
df = pd.read_csv('../data/loan_default_dataset.csv')
print(f'Shape: {df.shape}')
print(f'Memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB')
df.head(10)

In [ ]:
df.dtypes

In [ ]:
df.describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print()
print('Missing %:')
print((df.isnull().mean()*100).round(2))

## 2. Target Distribution

In [ ]:
counts = df['default'].value_counts()
print(counts)
print()
print(df['default'].value_counts(normalize=True).round(4))

fig, ax = plt.subplots(figsize=(5,4))
ax.bar(['No Default (0)','Default (1)'], counts.values, color=['#185FA5','#993C1D'], width=0.5)
for i,v in enumerate(counts.values):
    ax.text(i, v+1000, f'{v:,}\n({100*v/len(df):.1f}%)', ha='center', fontsize=11)
ax.set_title('Target Class Distribution', fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/00_target_distribution.png')
plt.show()

## 3. Feature Distributions

In [ ]:
features = [c for c in df.columns if c != 'default']

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, col in zip(axes.flatten(), features):
    df[col].dropna().hist(bins=50, ax=ax, color='#185FA5', edgecolor='white', linewidth=0.4)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('')
    skew = df[col].skew()
    ax.text(0.97, 0.95, f'skew={skew:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color='gray')
plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/01_feature_distributions.png', bbox_inches='tight')
plt.show()

## 4. Feature vs Target (Box Plots)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, col in zip(axes.flatten(), features):
    data0 = df[df['default']==0][col].dropna()
    data1 = df[df['default']==1][col].dropna()
    ax.boxplot([data0, data1], labels=['No Default','Default'],
               patch_artist=True,
               boxprops=dict(facecolor='#E6F1FB'),
               medianprops=dict(color='#185FA5', linewidth=2))
    ax.set_title(col, fontweight='bold')
plt.suptitle('Feature Distributions by Default Status', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/02_features_vs_target.png', bbox_inches='tight')
plt.show()

In [ ]:
print('Mean feature values by default status:')
df.groupby('default')[features].mean().T.rename(columns={0:'No Default',1:'Default'}).round(2)

## 5. Correlation Analysis

In [ ]:
plt.figure(figsize=(11, 8))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/03_correlation_heatmap.png')
plt.show()

In [ ]:
print('Point-biserial correlation with target (default):')
print('-'*55)
for col in features:
    valid = df[[col,'default']].dropna()
    r, p = pointbiserialr(valid[col], valid['default'])
    stars = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else ''))
    print(f'{col:30s}  r={r:+.4f}  p={p:.2e}  {stars}')

## 6. Segment Analysis

In [ ]:
df['credit_bucket'] = pd.cut(df['credit_score'],
    bins=[300,500,580,620,680,740,850],
    labels=['<500','500-580','580-620','620-680','680-740','>740'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

rate_by_credit = df.groupby('credit_bucket', observed=True)['default'].mean()
axes[0].bar(rate_by_credit.index, rate_by_credit.values, color='#993C1D')
axes[0].set_title('Default Rate by Credit Score Band', fontweight='bold')
axes[0].set_xlabel('Credit Score Range')
axes[0].set_ylabel('Default Rate')
axes[0].set_ylim(0, 0.6)
for i,(idx,v) in enumerate(rate_by_credit.items()):
    axes[0].text(i, v+0.01, f'{v:.1%}', ha='center', fontsize=10)

rate_by_derog = df.groupby('num_derogatory_marks')['default'].mean()
axes[1].bar(rate_by_derog.index, rate_by_derog.values, color='#854F0B')
axes[1].set_title('Default Rate by Derogatory Marks', fontweight='bold')
axes[1].set_xlabel('Number of Derogatory Marks')
axes[1].set_ylabel('Default Rate')
for i,(idx,v) in enumerate(rate_by_derog.items()):
    axes[1].text(i, v+0.005, f'{v:.1%}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/04_default_rate_segments.png')
plt.show()

## 7. Outlier Detection

In [ ]:
print('IQR-based outlier counts:')
print('-'*55)
for col in features:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    print(f'{col:30s}: {n_out:6,} outliers ({100*n_out/len(df):.2f}%)')

## 8. Key EDA Takeaways

- **Credit score** has the strongest negative correlation with default: lower score → higher risk  
- **Derogatory marks** have the strongest positive correlation: more marks → dramatically higher default rate  
- **Debt-to-income ratio** is right-skewed; high values strongly predict default  
- **Annual income** is log-normally distributed — will benefit from log transform or PowerTransformer  
- Missing values (~2%) are in: `annual_income`, `employment_years`, `credit_score`, `num_open_accounts` → impute with median  
- Class imbalance: 81.4% no-default vs 18.6% default → handle with `class_weight='balanced'`